# 서울시 상권 폐업위험 예측 — ① 데이터 준비와 통계 검정

> **핵심 질문** — 다음 분기에 점포 순감소(폐업 > 개업)로 전환될 상권 × 업종은 어디이며, 무엇이 위험을 만드는가

이 판본은 코드 없이 **출력물과 판단만** 담았습니다. 절마다 「핵심 한 줄 → 왜 이 분석을 했나 → 표·그림 → 읽는 법 → 해석」 순서입니다.

**01 ~ 10절** — 데이터와 전처리 판단 · 라벨 구조 · 기초 탐색 · 비교 기준 · 규모 고정 · 검정 방법 선택 · 가설 직접 검증 · 성향점수매칭 · 사후검정

이어지는 **② 예측 모델과 해석**은 [`상권_폐업위험_보고서_2_모델링.ipynb`](상권_폐업위험_보고서_2_모델링.ipynb) 입니다.


## 한 장 요약

| 절 | 제목 | 핵심 한 줄 |
|---|---|---|
| 01 | 분석 개요 | 다음 분기에 폐업 수가 개업 수보다 많아질 ‘상권×업종’을 한 분기 먼저 찾고, 그 위험이 어디에서 오는지 설명하는 것이 목표임. |
| 02 | 데이터와 전처리 판단 | 결측을 무조건 지우지 않고, 왜 비어 있는지부터 판단했음. 특히 매출 결측은 영세 상권의 특성을 담을 수 있어 그대로 정보로 남겼음. |
| 03 | 라벨의 구조 — y=0에 섞여 있는 두 상태 | 순감소가 아닌 y=0에는 “성장·유지”뿐 아니라 “아무 변화가 없었던 곳”도 많이 섞여 있음. 이 때문에 단순 모델은 위험보다 상권 규모를 먼저 배울 수 있음. |
| 04 | 기초 탐색과 상권유형별 차이 | 상권유형별 순감소율 차이는 통계적으로는 분명했지만, 실제 차이의 크기는 작았음. “유형 자체가 위험을 만듦”고 바로 결론낼 정도는 아니었음. |
| 05 | 비교 기준 — 모델이 넘어야 하는 선 | 예측모델은 무작위보다만 좋아서는 부족함. 점포 수 하나만으로도 상당히 맞힐 수 있기 때문에, 실제 기준선은 “점포 수 규칙”임. |
| 06 | 점포 수를 고정하면 무엇이 남는가 | 점포 수가 비슷한 상권끼리 비교하면 유형별 결론이 뒤집힌다 — 골목상권은 안전에서 위험으로, 발달상권은 위험에서 차이 없음으로. 규모가 상권유형과 위험 사이를 교란하고 있었음. |
| 07 | 검정 방법의 선택 — 정규성과 등분산성 | 데이터 분포가 정규분포와 거리가 있고 일부 집단은 분산도 달랐기 때문에, 평균 비교보다 순위 기반의 Kruskal-Wallis 검정을 주 검정으로 사용했음. |
| 08 | 가설 직접 검증 — 업종 × 상권유형 · 업종 × 자치구 | 같은 업종이라도 상권유형에 따라 위험이 달라지지만, 효과크기 기준으로 보면 실질적 차이가 있는 업종은 세 개뿐임. |
| 09 | 성향점수매칭 — 조건을 맞추고 다시 비교 | 한식음식점은 단순 비교에서는 더 위험해 보였지만, 규모와 입지가 비슷한 곳끼리 짝지으니 오히려 순감소 위험이 낮았음. |
| 10 | 사후검정 — 어느 상권유형이 실제로 다른가 | 전체적으로 차이가 있다는 것만으로는 부족해, Dunn 사후검정으로 상권유형을 두 개씩 비교했음. 전통시장과 골목상권만 뚜렷하게 구분되지 않았음. |
| 11 | 모델 선정 | 다섯 모델의 AP가 0.398~0.408에 몰려 있었고, 그 안에서 검증 AP가 가장 높은 랜덤포레스트를 대표 모델로 채택했음. |
| 12 | 선정 규모 — 몇 곳을 위험 대상으로 볼 것인가 | 모델은 많은 곳을 넓게 잡을 때보다, 실제 담당자가 볼 수 있는 상위 수십~수백 곳을 우선 선별할 때 가장 유용했음. |
| 13 | 테스트 최종 평가 — 2026년 1분기 결과 | 학습과 모델 선택에 쓰지 않은 홀드아웃 분기에서 **AP 0.413, Lift(향상도) 1.627**을 기록했음. 전체 순위 성능은 점포 수 기준과 비슷했지만, 상위 소수 구간에서는 모델이 더 높았음. |
| 14 | 예측 결과와 오차 — 무엇을 맞히고 무엇을 틀렸는가 | 「다음 분기에 줄어들 상권을 다 찾아내는 모델」로는 성립하지 않고, 「먼저 볼 50~300곳을 골라주는 도구」로는 성립한다. 맞힌 것과 틀린 것이 규모에 따라 갈리고, 놓친 위험은 작고 조용한 칸에 몰린다. |
| 15 | 두 단계로 나눈 모형 | 순감소를 “① 다음 분기에 개·폐업이 일어나는가”와 “② 일어났다면 감소 방향인가”로 나누자, 상권 규모 효과가 대부분 1단계에 모여 있음을 확인했음. |
| 16 | 두 확률로 나눈 상권 유형 | 최종 위험점수 하나로만 줄 세우지 않고 “변화가 자주 생기는가”와 “변하면 감소하는가”를 두 축으로 남기면 성격이 다른 네 유형을 구분할 수 있음. |
| 17 | 무엇이 위험과 관련되어 있는가 | 단일 모델에서는 규모·밀도 변수가 기여도의 3분의 2를 차지했지만, 사건과 방향을 분리하자 규모 영향은 주로 1단계에 남고 2단계에서는 4분의 1로 줄었음. |
| 18 | 위험도 공간 분석 | 지도에서는 위험등급이 높아질수록 실제 순감소율도 **13.4% → 42.1%**로 증가했음. 다만 자치구 경계나 역과의 거리 자체는 위험을 잘 설명하지 못했음. |
| 19 | 종합 결론 | 다음 분기 순감소를 완벽하게 예측한 것은 아니지만, 상위 소수 고위험 상권을 한 분기 먼저 선별하고 규모에 가려진 위험 구조를 분리해 설명할 수 있었음. |
| 20 | 비즈니스 제언 | 지원 대상은 업종이 아니라 상권 단위로, 그것도 「두 축」으로 골라야 함. 규모를 통제하지 않은 목록은 방향이 반대로 나옴. |


## 01  분석 개요

**핵심 한 줄 — 다음 분기에 폐업 수가 개업 수보다 많아질 ‘상권×업종’을 한 분기 먼저 찾고, 그 위험이 어디에서 오는지 설명하는 것이 목표임.**

### 왜 이 분석을 했나

연구 질문과 분석 단위를 먼저 고정해, 뒤의 통계검정과 예측모델이 같은 대상을 보고 있는지 명확히 했음.

### 핵심 결과

- 분석 단위는 상권 × 요식업 업종 × 분기임. 업종은 한식·중식·일식·양식·제과·패스트푸드·치킨·분식·호프·커피 등 10종임.
- 기간은 2023년 1분기~2025년 4분기이며, 2026년 1분기는 2025년 4분기의 “다음 분기 결과”를 만들기 위해서만 사용했음.
- 라벨은 “다음 분기에 폐업 수 > 개업 수인가”임. 매출 감소가 아니라 실제 점포 수의 순감소를 직접 잡기 위한 선택임.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 최종 표본 | 점포 5개 이상인 상권×업종×분기 67,475건 |
| 순감소 비율 | 25.5% |
| 예측 목표 | 다음 분기의 순감소 여부 |
| 핵심 산출물 | 위험순위 + 위험요인 + 상권 유형화 |

*표 1*


### 해석

따라서 이 보고서의 “폐업위험”은 개별 가게의 폐업확률이 아니라, 특정 상권의 특정 업종에서 다음 분기에 점포 수가 순감소할 가능성을 뜻함.



## 02  데이터와 전처리 판단

**핵심 한 줄 — 결측을 무조건 지우지 않고, 왜 비어 있는지부터 판단했음. 특히 매출 결측은 영세 상권의 특성을 담을 수 있어 그대로 정보로 남겼음.**

### 왜 이 분석을 했나

잘못된 결측 처리나 중복 변수를 넣으면 표본이 줄거나 같은 정보를 여러 번 세어 모델 해석이 왜곡될 수 있음.

### 핵심 결과

- 원천은 서울시 우리마을가게 상권분석서비스임. 매출은 카드 결제 기반 추정치, 유동인구는 생활인구, 점포는 사업자등록 자료를 사용함.
- “폐업”은 실제 문을 닫은 날짜가 아니라 사업자등록 말소 시점임. 따라서 실제 폐점 시점과 시차가 있을 수 있음.
- 집객시설의 빈칸은 시설이 없음을 뜻하는 자료 구조로 판단해 0으로 처리했음.
- 유동인구가 상주·직장 인구 정보를 이미 포함하므로 인구 변수를 중복해서 넣지 않았음. 직장인구 비중은 단독 판별력이 낮아(AUC 0.520) 제외했음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 매출 결측률 | 45.4% |
| 매출 결측 처리 | 행 삭제 대신 “매출이 관측되지 않음” 자체를 정보로 유지 |
| 기타 핵심 변수 | 면적·유동인구·점포 수·상권변화지표·운영개월 등은 결측이 거의 없음 |

*표 2*


|  | 컬럼 | 결측률 | 판정 |
|---|---|---|---|
| 0 | 엑스좌표_값 | 0.0% | OK |
| 1 | 영역_면적 | 0.0% | OK |
| 2 | 총_유동인구_수 | 0.0% | OK |
| 3 | 역세권 | 2.3% | OK |
| 4 | 상권_변화_지표 | 0.0% | OK |
| 5 | 당월_매출_금액 | 45.4% | 공변량 제외 권장 |
| 6 | 상권_전체점포 | 0.0% | OK |
| 7 | 운영_영업_개월_평균 | 0.0% | OK |
| 8 | 운영개월_서울대비 | 0.0% | OK |
| 9 | 외식물가지수 | 0.0% | OK |
| 10 | 외식지출전망 | 0.0% | OK |

*표 3*

읽는 법 — 결측률 5% 이하는 그대로 쓰고 30%를 넘으면 공변량에서 빼는 기준으로 「판정」 열을 붙였음. 매출 계열만 주의 구간에 들어감.

기준 — 결측률 5% 이하는 그대로 사용 · 5~30% 주의 · 30% 초과는 공변량에서 제외 검토. 관례적 눈금이며 이 표에서 제외 대상은 없음.


### 해석

핵심은 “결측=오류”로 단정하지 않은 것임. 카드 결제가 충분히 잡히지 않는 소규모 상권이라는 특성이 결측에 들어 있을 수 있으므로, 결측 행을 통째로 버리면 오히려 작은 상권을 체계적으로 잃게 됨.



## 03  라벨의 구조 — y=0에 섞여 있는 두 상태

**핵심 한 줄 — 순감소가 아닌 y=0에는 “성장·유지”뿐 아니라 “아무 변화가 없었던 곳”도 많이 섞여 있음. 이 때문에 단순 모델은 위험보다 상권 규모를 먼저 배울 수 있음.**

### 왜 이 분석을 했나

작은 상권은 안전해서가 아니라 점포 수가 적어서 한 분기 동안 개업·폐업이 아예 관측되지 않을 가능성이 높음.

### 핵심 결과

- 전체의 43.9%는 다음 분기에 개업도 폐업도 없는 “변화 없음”이었음.
- y=0인 50,285건 중 58.9%가 “변화 없음”이었음. 즉 y=0을 곧바로 “안전”이라고 해석하면 안 됨.
- 점포 수가 가장 적은 구간에서는 변화 없음이 69.3%였지만, 가장 큰 구간에서는 7.9%까지 떨어졌음. 같은 방향으로 순감소율은 15.1%에서 40.3%로 올라갔음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 변화 없음 | 29,633건 · 43.9% |
| 변화는 있었지만 순감소 아님 | 20,652건 · 30.6% |
| 순감소 | 17,190건 · 25.5% |
| 점포 규모 효과 | 평균 5.4개 구간: 변화 없음 69.3% → 평균 65.5개 구간: 7.9% |

*표 4*


|  | 건수 | 비율 |
|---|---|---|
| 변화 없음 | 29633 | 0.439 |
| 변화 있었으나 줄지 않음 | 20652 | 0.306 |
| 순감소 | 17190 | 0.255 |

*표 5*

읽는 법 — 다음 분기 상태를 셋으로 나눈 비율임. y=0 은 위 두 줄(변화 없음 + 줄지 않음)의 합임.

기준 — y=0 은 「변화 없음」과 「변화 있었으나 줄지 않음」의 합. 판정 기준이 아니라 라벨 구성의 분해임.


|  | 칸수 | 평균점포 | 변화없음비율 | 순감소율 |
|---|---|---|---|---|
| 1분위 | 15520 | 5.44 | 0.693 | 0.151 |
| 2분위 | 14018 | 7.86 | 0.599 | 0.193 |
| 3분위 | 12139 | 11.73 | 0.465 | 0.243 |
| 4분위 | 12718 | 19.56 | 0.298 | 0.308 |
| 5분위 | 13080 | 65.47 | 0.079 | 0.403 |

*표 6*

읽는 법 — 1분위가 점포가 가장 적은 구간임. 「변화없음비율」 열을 위에서 아래로 읽으면 규모가 커질 때 「아무 일도 없음」이 어떻게 줄어드는지 보임.

기준 — 규모가 커질 때 「변화없음비율」이 줄고 「순감소율」이 오르면 라벨이 규모에 끌려간다는 신호.


### 해석

따라서 아무 조정 없이 학습하면 모델은 “어떤 조건에서 쇠퇴하는가”보다 “어디에서 사건이 일어날 만큼 점포가 많은가”를 학습할 수 있음. 이후의 규모 통제와 2단계 모형은 이 문제를 해결하기 위한 장치임.



## 04  기초 탐색과 상권유형별 차이

**핵심 한 줄 — 상권유형별 순감소율 차이는 통계적으로는 분명했지만, 실제 차이의 크기는 작았음. “유형 자체가 위험을 만듦”고 바로 결론낼 정도는 아니었음.**

### 왜 이 분석을 했나

표본이 수만 건이면 아주 작은 차이도 p-value가 매우 작게 나옴. 그래서 “차이가 존재하는가”와 “차이가 큰가”를 함께 봐야 함.

### 핵심 결과

- 상권유형 × 순감소 카이제곱 검정은 유의했지만 Cramér’s V=0.071로 효과크기는 작았음.
- Kruskal-Wallis도 유의했지만 ε²=0.0050으로 실제 차이는 크지 않았음.
- 변수 간 상관은 일부 높았지만 |r|>0.8인 쌍은 없었음. 가장 큰 상관은 업종밀도와 업종점포비중(r=0.753)이었음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 통계적 차이 | χ²(3)=337.9, p<0.001 |
| 효과크기 | Cramér’s V=0.071 · ε²=0.0050 → 작음 |
| 해석 주의 | 상권유형 안에 규모와 업종 구성이 섞여 있으므로 추가 통제가 필요 |

*표 7*

기준 — 효과크기 기준이 처음 나오는 자리임 — **Cramér's V 는 0.1 작음 / 0.3 중간 / 0.5 큼**(자유도 1 기준), **ε² 는 0.01 작음 / 0.06 중간 / 0.14 큼**임. 표본이 수만 건이면 p 는 거의 항상 유의하게 나오므로 이 절부터 판정은 효과크기로 함.


![그림 1](img/fig_04_01.png)


*그림 1*

읽는 법 — 왼쪽이 유형별, 오른쪽이 업종별 순감소율이고 점선이 전체 평균임.


![그림 2](img/fig_04_02.png)


*그림 2*

읽는 법 — 구간 안에서의 비율이라 막대 하나의 합이 100%임. 회색이 「변화 없음」이고, 왼쪽으로 갈수록 회색이 커지는 것이 규모 편향의 통로임.


![그림 3](img/fig_04_03.png)


*그림 3*

읽는 법 — 공변량끼리 얼마나 「같은 것을 측정하고 있는가」임. 붉을수록 강한 양의 상관이고, 0.7 안팎으로 붙어 있는 「업종 밀도 ↔ 업종 점포 비중」과 「유동인구 ↔ 면적」은 각각 한 덩어리로 읽어야 함.


### 해석

이 단계의 결론은 “상권유형별 차이가 아예 없음”도, “상권유형이 원인임”도 아님. 차이는 보이지만 그 크기가 작으므로, 점포 수와 업종 구성을 분리한 뒤 다시 확인해야 함.



## 05  비교 기준 — 모델이 넘어야 하는 선

**핵심 한 줄 — 예측모델은 무작위보다만 좋아서는 부족함. 점포 수 하나만으로도 상당히 맞힐 수 있기 때문에, 실제 기준선은 “점포 수 규칙”임.**

### 왜 이 분석을 했나

큰 상권일수록 개업·폐업 사건이 더 자주 발생함. 이 규모 효과를 이기지 못하면 복잡한 모델을 쓸 이유가 없음.

### 핵심 결과

- 무작위로 고르면 정밀도는 전체 순감소율과 같은 25.5%임.
- 직전 분기 상태를 그대로 쓰면 정밀도 28.4%, Lift(향상도) 1.11로 개선 폭이 작았음.
- 서울시 상권변화지표(HL)는 정밀도 24.4%, Lift(향상도) 0.956으로 다음 분기 예측 목적에서는 무작위보다 낮았음.
- 점포 수가 많은 순으로 같은 수를 고르면 정밀도 39.4%, Lift(향상도) 1.54로 가장 강한 단순 기준이 됐음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 무작위 | 정밀도 25.5% · Lift(향상도) 1.00 |
| 직전 분기 | 28.4% · 1.11 |
| 서울시 상권변화지표(HL) | 24.4% · 0.956 |
| 점포 수 | 39.4% · 1.54 |

*표 8*


|  | 기준선 | 위험으로 찍은 수 | 정밀도 | 재현율 | Lift |
|---|---|---|---|---|---|
| 0 | 무작위(전부 위험) | 67475 | 0.255 | 1 | 1 |
| 1 | 직전 분기 | 15682 | 0.284 | 0.259 | 1.11 |
| 2 | 상권변화지표(HL) | 12608 | 0.244 | 0.179 | 0.956 |
| 3 | 점포 수(상위 15,682칸) | 15682 | 0.394 | 0.359 | 1.54 |

*표 9*

읽는 법 — **기저율은 그 분기에 실제로 순감소한 칸의 비율**임 — 전체 표본 25.5%, 테스트 분기 26.0%. 아무 정보 없이 한 칸을 찍었을 때 맞을 확률이고, **Lift(향상도) = 정밀도 ÷ 기저율**이라 1.0 이 무작위임. 첫 줄 「무작위」는 67,475칸을 전부 찍는 설정이라 재현율이 정의상 1.0 이므로, 찍은 수가 다른 규칙끼리 재현율을 견주면 안 됨. 모델이 넘어야 하는 선은 이 표에서 가장 높은 값(점포 수)임.

기준 — 정밀도·재현율·Lift(향상도) 가 처음 나오는 자리임 — **정밀도는 위험으로 찍은 칸 중 실제 순감소였던 비율**, **재현율은 실제 순감소 칸 중 잡아낸 비율**이고 둘은 같은 작동점에서 맞바꾸는 값임. **Lift(향상도) = 정밀도 ÷ 기저율이라 1.0 이 무작위이고 1 미만은 무작위보다 못함.** 모델이 넘어야 하는 선은 이 표에서 가장 높은 값(점포 수 1.545).


### 해석

따라서 모델 성능을 “무작위보다 좋다”라고만 말하면 과장임. 이 연구에서 모델이 실제로 넘어야 할 비교선은 가장 강한 단순 규칙인 점포 수 기준임.



## 06  점포 수를 고정하면 무엇이 남는가

**핵심 한 줄 — 점포 수가 비슷한 상권끼리 비교하면 유형별 결론이 뒤집힌다 — 골목상권은 안전에서 위험으로, 발달상권은 위험에서 차이 없음으로. 규모가 상권유형과 위험 사이를 교란하고 있었음.**

### 왜 이 분석을 했나

전체를 한꺼번에 비교하면 골목상권이 상대적으로 작은 상권에 많이 있다는 사실과 순감소 위험이 섞임.

### 핵심 결과

- 점포 수 5분위 안에서 다시 비교하면 점포 수 규칙의 Lift(향상도)가 대부분 1에 가까워졌음. 전체에서 좋았던 성능의 상당 부분이 “큰 상권 고르기”였다는 뜻임.
- 규모를 통제하지 않은 단순 비교에서는 골목상권 OR=0.813로 더 안전해 보였음.
- 점포 수 구간을 통제한 CMH 검정에서는 공통 OR=1.061, p=0.0017로 방향이 반대로 바뀌었음.
- 다만 규모별 효과가 같지는 않았음. 작은·중간 구간 OR은 1.09~1.17이었지만 가장 큰 구간은 0.89였음. **같은 검정을 네 유형 각각에 돌리면 발달상권도 1.340 → 0.992로 뒤집힌다**(다만 p=0.689로 비유의). 전통시장만 층별 오즈비가 균일해(Woolf p=0.180) 하나의 값 0.896으로 요약할 수 있음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 단순 비교 | 골목상권 OR 0.813 → 더 안전해 보임 |
| 규모 통제 후 | CMH 공통 OR 1.061 [1.023, 1.101] |
| 규모별 차이 | Woolf p=0.0000166 → 하나의 OR로 요약하기 어려움 |

*표 10*


|  | 칸수 | 평균점포 | 기저율 | 직전 분기 Lift | 상권변화지표 Lift | 점포 수 Lift | 점포 수 범위 |
|---|---|---|---|---|---|---|---|
| 1분위 | 15520 | 5.4 | 0.151 | 0.947 | 1.01 | **1.13** | 5 ~ 6개 |
| 2분위 | 14018 | 7.9 | 0.193 | 0.963 | 1.05 | **1.15** | 7 ~ 9개 |
| 3분위 | 12139 | 11.7 | 0.243 | 1 | 1.01 | **1.09** | 10 ~ 14개 |
| 4분위 | 12718 | 19.6 | 0.308 | 0.955 | 1.06 | **1.1** | 15 ~ 26개 |
| 5분위 | 13080 | 65.5 | 0.403 | 1.01 | 0.969 | **1.06** | 27개 이상 |
| **전체(층 무시)** | **67,475** | **21.4** | **0.255** | **1.11** | **0.956** | **1.545** | 5개 이상 전체 |

*표 11*

읽는 법 — 층 안에서 각 기준선을 다시 측정한 값임. 각 층에서 「직전 분기 순감소 칸수」만큼을 점포 수 많은 순으로 찍어 정밀도를 그 층의 기저율로 나눈 것임. **1분위 1.13 > 5분위 1.06 이 규모가 작은 쪽에서 점포 수가 더 유용하다는 뜻은 아님** — 층마다 기저율이 달라 Lift(향상도) 의 상한이 다르기 때문임(1분위는 최대 6.6배, 5분위는 2.5배). 또 5분위에서는 층의 40%를 찍게 되어 정밀도가 기저율에 수렴함. 읽어야 할 것은 층끼리의 비교가 아니라 **전체 표본에서 1.54 였던 값이 층 안에서 1.06~1.15 로 내려앉았다는 사실**임.

기준 — 층 안에서 Lift(향상도) 가 1 근처로 내려가면 그 규칙이 위험이 아니라 규모를 고르고 있었다는 뜻. 전체 표본 값과의 낙차가 「규모를 고른 몫」의 크기임.


**층(점포 수 5분위 · 괄호는 그 층의 점포 수 범위) × 네 상권유형 — 순감소율 % · 괄호는 그 유형의 칸수**

| 층 | 골목상권 | 발달상권 | 전통시장 | 관광특구 | 층 전체 |
|---|---|---|---|---|---|
| 1분위 (5~6개) | **15.8 (10,254)** | 12.8 (2,683) | 14.9 (2,577) | — (6) | 15,520 |
| 2분위 (7~9개) | **20.0 (8,518)** | 18.3 (3,368) | 18.3 (2,126) | — (6) | 14,018 |
| 3분위 (10~14개) | **25.2 (6,599)** | 23.5 (3,955) | 22.2 (1,578) | — (7) | 12,139 |
| 4분위 (15~26개) | **31.9 (5,667)** | 30.4 (5,405) | 29.1 (1,468) | 26.4 (178) | 12,718 |
| 5분위 (27개 이상) | 38.2 (3,609) | 42.0 (7,600) | 35.6 (1,372) | 42.1 (499) | 13,080 |
| 전체(층 무시) | **23.6 (34,647)** | **29.2 (23,011)** | 22.4 (9,121) | 37.8 (696) | 67,475 |

*표 12*

기준 — 판정 기준이 아니라 서술. 층을 무시한 순서와 층 안의 순서가 갈리는지를 보는 표임.


**네 유형 각각 「그 유형 대 나머지」 CMH**

| 상권유형 | 칸수 | 단순 오즈비 | MH 공통 오즈비 | 규모 통제 후 | 95% CI | CMH p | Woolf p |
|---|---|---|---|---|---|---|---|
| 골목상권 | 34,647 | 0.813 | **1.061** | ▲ +0.248 | [1.023, 1.101] | 0.0017 | 0.0000166 |
| 발달상권 | 23,011 | 1.340 | **0.992** | ▼ −0.348 | [0.954, 1.031] | 0.689 | 0.0000000187 |
| 전통시장 | 9,121 | 0.821 | **0.896** | ▲ +0.075 | [0.849, 0.945] | 0.00006 | **0.180** |
| 관광특구 | 696 | 1.789 | 1.023 | ▼ −0.766 | [0.873, 1.197] | 0.812 | 0.247 |

*표 13*

읽는 법 — **화살표 표기가 처음 나오는 자리임** — ▲ 는 값이 올라감, ▼ 는 내려감을 뜻하고 **색은 방향만 나타낼 뿐 좋고 나쁨과는 무관함.** 이 표에서는 ▲ 가 규모를 통제한 뒤 오즈비가 올라갔다는 뜻이므로 **위험이 커진 쪽**이고, 13절 「모델 − 점포 수」 열에서는 ▲ 가 모델이 기준선보다 높다는 뜻이라 **반대로 좋은 쪽**임. **표마다 무엇이 올라간 것인지 열 이름을 함께 읽어야 함.**

기준 — **오즈비는 1 이 「차이 없음」** — 95% CI 가 1 을 포함하지 않으면 유의. 크기 관행은 1.5 작음 / 2.5 중간 / 4.3 큼(확률로는 기저율 26% 기준 1.5 ≈ +8.5%p). **Woolf p ≥ 0.05 일 때만 공통 오즈비를 하나의 값으로 요약할 수 있음.**


### 해석

이 결과는 Simpson의 역설에 해당함. 실무적으로는 “골목상권이 더 위험하다/안전함”처럼 한 문장으로 묶기보다, 점포 규모를 나눠서 말해야 함.

**부호 반전은 골목상권과 발달상권 두 곳에서 나타나고, 두 유형은 서로 거울상이다** — 골목은 작은 층에서 불리(1.17)하고 큰 층에서 유리(0.886)한데 발달은 정확히 반대(0.793 → 1.187)다. **즉 남는 것은 유형 자체가 아니라 「규모 × 유형」의 상호작용**이며, 하나의 값으로 요약할 수 있는 유형은 전통시장뿐임.



## 07  검정 방법의 선택 — 정규성과 등분산성

**핵심 한 줄 — 데이터 분포가 정규분포와 거리가 있고 일부 집단은 분산도 달랐기 때문에, 평균 비교보다 순위 기반의 Kruskal-Wallis 검정을 주 검정으로 사용했음.**

### 왜 이 분석을 했나

검정 방법은 결과가 잘 나오는 것을 고르는 것이 아니라, 데이터가 어떤 분포를 가지는지 확인한 뒤 정해야 함.

### 핵심 결과

- 0/1 라벨 자체에 정규성 검정을 하는 대신, 각 상권×업종의 여러 분기 순감소 비율을 만들어 연속값으로 비교했음.
- 상권유형 4/4, 업종 10/10 그룹에서 정규성 검정이 기각됐음. 다만 이 변수는 1/12 단위의 이산량이므로, 이 결과는 참고로 두고 이산 데이터에 맞는 적합도 검정을 따로 했음.
- 등분산성도 상권유형과 업종 모두에서 위배됐다(Levene 중앙값 기준).
- 전체 분포는 왜도 +0.592로 오른쪽 꼬리가 길었고, 순감소가 한 번도 없었던 조합도 7.1% 있었음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 정규성 | 상권유형 4/4 · 업종 10/10 위배 (참고값) |
| 등분산성 | 상권유형·업종 모두 위배 |
| 주 검정 | Kruskal-Wallis |
| 시각 확인 | Q-Q plot + Boxplot |

*표 14*


|  | 그룹 수 | 정규성 위배 | 정규성 성립 그룹 | Levene 통계량 | Levene p | 등분산성 |
|---|---|---|---|---|---|---|
| 상권유형 | 4 | 4/4 | 없음 | 16 | 2.3e-10 | 위배 |
| 서비스_업종_코드_명 | 10 | 10/10 | 없음 | 10.83 | 0 | 위배 |

*표 15*

읽는 법 — 그룹별 정규성 위배 개수와 등분산성(Levene) 결과임. 정규성 열은 관례상 남긴 참고값이고, **판정은 아래 적합도 검정으로 함** — 이 변수는 이산량이라 연속 분포를 전제하는 정규성 검정이 표본만 커지면 자동으로 기각되기 때문임.

기준 — **Levene p ≥ 0.05 면 등분산 성립.** 정규성 열은 관례상 남긴 참고값이고 판정은 아래 적합도 검정으로 함(이 변수는 이산량이라 연속 분포를 전제하는 검정이 표본만 커지면 기각됨).


![그림 4](img/fig_07_04.png)


*그림 4*

읽는 법 — 상자 높이가 그룹마다 다른 것이 등분산성 위배의 모습이고, 위쪽에 흩어진 점들이 오른쪽 꼬리임.


![그림 5](img/fig_07_05.png)


*그림 5*

읽는 법 — 왼쪽은 칸의 12분기 순감소 비율 분포임 — **0 에 봉우리가 서고 오른쪽으로 꼬리가 길다**(왜도 +0.59). 구간은 값의 최소 단위인 1/12 로 잡았음. 오른쪽은 같은 자료를 「12분기 중 순감소한 분기 수」로 세고 **정규(이산화) 기대도수를 겹친 것**임 — 관측이 왼쪽에서 더 두껍고 오른쪽 꼬리가 길어 정규분포와 어긋나는 방향이 드러남. 다른 후보 분포와의 비교는 다음 그림에서 다룸.


![그림 6](img/fig_07_06.png)


*그림 6*

읽는 법 — **상권유형과 업종을 따로 떼어** 카이제곱 적합도 검정 p 를 로그 눈금에 찍은 것임. 점선 오른쪽이면 그 분포로 설명된다는 뜻임. **정규·이항은 왼쪽에 몰리고 베타이항만 오른쪽에 있음.** 관광특구는 58칸뿐이라 셋 모두 통과하는데, 표본이 작아 어떤 분포도 기각하지 못하는 경우임.


**상권유형별 분포 모양과 정규 적합도 검정**

| 구분 | 칸수 | 왜도 | 첨도 | 0인 칸 % | Q-Q R² | 정규 적합 p |
|---|---|---|---|---|---|---|
| 발달상권 | 1,815 | 0.457 | −0.024 | 3.8 | 0.961 | **8e-08** |
| 골목상권 | 2,393 | 0.522 | 0.163 | 7.6 | 0.949 | **2e-12** |
| 전통시장 | 632 | 0.748 | 0.898 | 9.0 | 0.936 | **0.0002** |
| 관광특구 | 58 | 0.783 | 1.135 | 0.0 | 0.935 | **0.562** |

*표 16*

읽는 법 — **관광특구만 통과(p=0.562)하는데 이는 정규분포라는 증거가 아니라 「기각할 힘이 없다」는 뜻임** — 58칸뿐이어서 기대도수 5 미만 구간이 대부분 병합되고 자유도가 거의 남지 않음. 실제로 모양 지표는 네 유형 중 가장 나쁨(왜도 0.783 · 첨도 1.135 · Q-Q R² 0.935). **표본이 작으면 어떤 분포도 기각되지 않는다**는 것을 보여주는 자리이므로, 판정은 항상 칸수와 함께 읽어야 함.

기준 — **적합도 검정은 「그 분포를 따른다」가 귀무가설이므로 p ≥ 0.05 면 적합.** 왜도는 0 이 대칭이고 |왜도| 0.5 이상이면 뚜렷한 치우침, Q-Q R² 는 1.0 이 완전 정규이며 0.98 아래는 눈에 보이는 이탈로 읽음.


**업종별 분포 모양과 정규 적합도 검정**

| 구분 | 칸수 | 왜도 | 첨도 | 0인 칸 % | Q-Q R² | 정규 적합 p |
|---|---|---|---|---|---|---|
| 커피-음료 | 911 | **0.329** | −0.377 | 4.0 | **0.959** | **0.00217** |
| 양식음식점 | 287 | 0.331 | −0.446 | 8.0 | 0.961 | **0.0664** |
| 분식전문점 | 523 | 0.393 | −0.120 | 8.0 | 0.953 | **0.132** |
| 중식음식점 | 253 | 0.398 | −0.322 | 8.7 | 0.946 | **0.0522** |
| 한식음식점 | 1,257 | 0.406 | 0.127 | 5.6 | 0.961 | **0.0101** |
| 제과점 | 257 | 0.444 | −0.154 | 6.6 | 0.948 | **0.183** |
| 일식음식점 | 259 | 0.628 | 0.071 | 5.4 | 0.947 | **0.0195** |
| 치킨전문점 | 261 | 0.667 | 0.324 | 12.3 | 0.929 | **0.0491** |
| 호프-간이주점 | 630 | 0.669 | 0.055 | 6.0 | 0.947 | **2e-06** |
| 패스트푸드점 | 260 | **0.851** | 0.722 | 4.6 | **0.924** | **0.00108** |

*표 17*

읽는 법 — **왜도·첨도·Q-Q R² 는 표본 크기에 흔들리지 않는 모양 지표**이고, 마지막 열은 **이산 데이터에 맞는 카이제곱 적합도 검정**임(12분기 완전 관측 4,898칸에서 순감소 횟수 0~12 의 관측 도수를 정규분포의 기대도수와 비교, 기대도수 5 미만 구간은 이웃과 합침). p 가 0.05 보다 크면 정규분포로 설명된다는 뜻이고, **열 업종 중 넷만 통과함**.

기준 — 위 표와 같은 기준. p 가 작으면 「정규분포가 아님」이고, 표본이 작으면 무엇이든 통과하므로 항상 칸수와 함께 읽음.


**같은 자료를 베타이항에 맞추면 — 칸마다 확률이 다른 것을 허용한 경우**

| 구분 | 칸수 | 왜도 | 첨도 | 0인 칸 % | Q-Q R² | 베타이항 적합 p |
|---|---|---|---|---|---|---|
| 유형 · 발달상권 | 1,815 | 0.457 | −0.024 | 3.8 | 0.961 | 0.653 |
| 유형 · 골목상권 | 2,393 | 0.522 | 0.163 | 7.6 | 0.949 | 0.362 |
| 유형 · 전통시장 | 632 | 0.748 | 0.898 | 9.0 | 0.936 | 0.268 |
| 유형 · 관광특구 | 58 | 0.783 | 1.135 | 0.0 | 0.935 | 0.592 |
| 업종 · 커피-음료 | 911 | 0.329 | −0.377 | 4.0 | 0.959 | 0.415 |
| 업종 · 양식음식점 | 287 | 0.331 | −0.446 | 8.0 | 0.961 | 0.349 |
| 업종 · 분식전문점 | 523 | 0.393 | −0.120 | 8.0 | 0.953 | 0.352 |
| 업종 · 중식음식점 | 253 | 0.398 | −0.322 | 8.7 | 0.946 | 0.143 |
| 업종 · 한식음식점 | 1,257 | 0.406 | 0.127 | 5.6 | 0.961 | 0.0695 |
| 업종 · 제과점 | 257 | 0.444 | −0.154 | 6.6 | 0.948 | 0.907 |
| 업종 · 일식음식점 | 259 | 0.628 | 0.071 | 5.4 | 0.947 | 0.779 |
| 업종 · 치킨전문점 | 261 | 0.667 | 0.324 | 12.3 | 0.929 | 0.497 |
| 업종 · 호프-간이주점 | 630 | 0.669 | 0.055 | 6.0 | 0.947 | 0.479 |
| 업종 · 패스트푸드점 | 260 | 0.851 | 0.722 | 4.6 | 0.924 | 0.255 |

*표 18*

읽는 법 — **베타이항은 「칸마다 순감소 확률이 다르다」를 허용하는 분포임.** 이항은 모든 칸이 같은 확률 p 로 12번 시도한다고 보지만, 베타이항은 칸마다 p 가 흩어져 있다고 보고 그 흩어짐까지 모수로 추정함. **정규 적합에서 기각됐던 유형·업종이 여기서는 대부분 통과함** — 관측 도수의 모양이 정규에서 벗어난 이유가 「칸 사이 위험도 차이」였다는 뜻임.

기준 — 같은 기준(p ≥ 0.05 면 적합). 베타이항이 통과한다는 것은 「칸마다 확률이 다름」를 허용해야 관측 도수가 설명된다는 뜻임.


### 해석

따라서 ANOVA는 참고값으로 두고, 정규성 가정을 덜 요구하는 Kruskal-Wallis를 중심으로 결론을 냈음. 이는 사후에 결과를 맞춘 선택이 아니라 사전 가정 점검에서 나온 선택임.

**이 변수는 연속량이 아니라 「12분기 중 몇 번」이라는 이산량임.** 그래서 연속 분포를 전제하는 검정 대신 **이산 데이터의 표준 방법인 카이제곱 적합도 검정**을 썼다 — 순감소 횟수 0~12 의 관측 도수를 후보 분포의 기대도수와 비교하고, 기대도수 5 미만 구간은 이웃과 합쳤음.

**열 업종 중 넷만 통과한다** — 제과점(0.183)·분식전문점(0.132)·양식음식점(0.066)·중식음식점(0.052)이고, 나머지 여섯은 기각임. 상권유형에서는 관광특구(0.562)만 통과하는데 58칸뿐이라 어떤 분포도 기각하지 못하는 경우다.

참고로 같은 자료를 **베타이항**(칸마다 위험 확률이 다른 것을 허용하는 분포)에 맞추면 전체 p=0.216 으로 통과함. 「모든 칸이 같은 확률을 갖음」가 성립하지 않는다는 뜻이고, 그 이질성을 찾는 것이 이 과제의 목표임.

모양 지표로는 왜도 기준 **커피-음료가 가장 대칭(0.329 · Q-Q R² 0.959)**이고 **패스트푸드점이 가장 치우쳐 있다(0.851 · 0.924)**. Q-Q R² 는 1.0 이 완전 정규이고 0.98 아래면 눈에 보이는 이탈이므로 열 업종 모두 이탈 구간이며, 이 값은 판정용이 아니라 **업종 간 순서를 매기는 용도**다.



## 08  가설 직접 검증 — 업종 × 상권유형 · 업종 × 자치구

**핵심 한 줄 — 같은 업종이라도 상권유형에 따라 위험이 달라지지만, 효과크기 기준으로 보면 실질적 차이가 있는 업종은 세 개뿐임.**

### 왜 이 분석을 했나

전체 업종을 섞으면 “업종 효과”와 “상권유형 효과”가 붙음. 그래서 업종을 하나씩 고정한 뒤 상권유형 차이를 따로 검정했음.

### 핵심 결과

- 10개 업종 중 8개가 Bonferroni 보정 후에도 p 기준으로는 상권유형별 차이를 보였음. 다만 **표본이 크면 p는 거의 항상 유의하므로 판정은 효과크기(ε²)로 함.**
- **ε² 기준은 0.01 작음 / 0.06 중간 / 0.14 큼임.** 중간 이상은 호프-간이주점 0.1202 · 커피-음료 0.0652 · 한식음식점 0.0607 **세 업종뿐**이고, 나머지 다섯(양식 0.0483 · 분식 0.0481 · 일식 0.0448 · 제과 0.0318 · 중식 0.0300)은 「작음」 구간에 머묾.
- 패스트푸드점은 ε²=0.0161로 작고 p도 보정 기준을 통과하지 못했음.
- 치킨전문점은 ε²=0.0002로 차이가 사실상 없고 통계적으로도 유의하지 않았음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 상권유형 차이 | 10개 업종 중 8개 유의 |
| 가장 큰 효과 | 호프·간이주점 ε²=0.1202 |
| 거의 차이 없음 | 치킨전문점 ε²=0.0002 |
| 평균 효과크기 | 상권유형 0.0465 > 자치구 0.0211 |

*표 19*


|  | 그룹수 | H | n | p-value | 유의 | eps2 |
|---|---|---|---|---|---|---|
| 호프-간이주점 | 3 | 86.9 | 709 | 0 | True | **0.1202** |
| 커피-음료 | 3 | 67.4 | 1006 | 2.0e-15 | True | **0.0652** |
| 한식음식점 | 3 | 81 | 1306 | 0 | True | **0.0607** |
| 양식음식점 | 3 | 18.8 | 352 | 8.1e-05 | True | 0.0483 |
| 분식전문점 | 3 | 32.5 | 637 | 8.8e-08 | True | 0.0481 |
| 일식음식점 | 3 | 15.6 | 307 | 4.0e-04 | True | 0.0448 |
| 제과점 | 3 | 12.9 | 347 | 0.0015 | True | 0.0318 |
| 중식음식점 | 3 | 11 | 302 | 0.0041 | True | 0.03 |
| 패스트푸드점 | 3 | 7.1 | 322 | 0.0281 | False | 0.0161 |
| 치킨전문점 | 3 | 2.1 | 341 | 0.3552 | False | **2.0e-04** |

*표 20*

읽는 법 — 업종을 하나로 고정하고 그 안에서 상권유형 간 차이를 검정한 결과임. **색이 칠해진 행은 p 가 아니라 효과크기 ε² ≥ 0.06(중간 이상)인 업종**임 — 표본이 커서 p 는 대부분 유의하게 나오므로 판정 기준을 효과크기로 두었음. ε² 는 0.01 작음 / 0.06 중간 / 0.14 큼으로 읽음.

기준 — **판정은 p 가 아니라 효과크기 ε².** 0.01 작음 / 0.06 중간 / 0.14 큼이며, 색은 0.06 이상에만 칠함. 표본이 커서 p 는 대부분 유의하게 나오므로 p 단독으로는 판정하지 않음.


### 해석

**「모든 업종이 입지에 민감함」는 결론은 틀림.** p 만 보면 8개가 유의하지만, 실질적 크기로 보면 **입지가 위험을 가르는 업종은 호프-간이주점·커피-음료·한식음식점 세 개**다. 나머지는 유의하더라도 그 차이가 작아 정책 근거로 쓰기 어려움.

업종별로 처방이 갈려야 한다는 뜻임 — 세 업종은 입지 조정·상권 선택이 효과를 낼 수 있고, 치킨전문점처럼 ε²가 0에 가까운 업종은 어디에 있든 위험 수준이 비슷하므로 입지와 무관한 지원(배달 구조·비용 구조)이 맞음.



## 09  성향점수매칭 — 조건을 맞추고 다시 비교

**핵심 한 줄 — 한식음식점은 단순 비교에서는 더 위험해 보였지만, 규모와 입지가 비슷한 곳끼리 짝지으니 오히려 순감소 위험이 낮았음.**

### 왜 이 분석을 했나

업종별 상권 규모가 너무 달라 그대로 비교하면 “업종 차이”가 아니라 “큰 상권에 많이 들어가 있는가”를 비교하게 됨.

처리군은 한식음식점으로 정했음. **표본이 가장 크고(15,677칸) 네 상권유형에 고르게 분포하기 때문**임.

### 핵심 결과

- 한식음식점을 처리군으로 정하고 점포 수·유동인구·면적·집객시설을 맞춰 1:1 매칭했음. **표본은 전체 상권 4유형 67,475칸이고, 짝은 분기와 상권유형을 정확히 일치시킨 블록 안에서만 찾았음.**
- 한식음식점 15,677칸 중 **10,030쌍**이 매칭됐고(매칭률 64.0%), 매칭 후 모든 조건 변수의 |SMD|가 0.1 아래로 내려가 비교 가능한 수준이 됐다(최대 0.015).
- **매칭 전 한식음식점은 +1.85%p 더 위험해 보였지만, 매칭 후 ATT는 −6.53%p로 방향이 반대로 바뀌었음.**
- McNemar 검정(불일치 쌍 3,929개), 상권 단위 클러스터 부트스트랩 신뢰구간, 캘리퍼·복원·유형 미고정을 바꾼 **여섯 설계가 모두 같은 방향**을 보였다(−6.47 ~ −7.65%p).
- **유형별로 나누어도 네 유형 모두 음수다** — 골목상권 −5.39 · 전통시장 −8.51 · 발달상권 −9.30%p(관광특구는 쌍 50개로 참고값). 한 유형의 산물이 아님.
- **같은 설계를 10개 업종에 모두 돌리면 「업종 자체의 위험」은 대부분 사라진다** — 규모를 맞춘 뒤에도 유의하게 남는 것은 치킨전문점 +3.35%p · 호프-간이주점 +2.34%p · 패스트푸드점 +1.97%p 세 개이고, 한식음식점은 −6.52%p 로 오히려 안전함. 중식·커피·제과·분식·양식은 유의하지 않음.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 표본 | 전체 상권 4유형 67,475칸 (특정 유형을 떼지 않음) |
| 매칭 쌍 | 10,030쌍 · 처리군 매칭률 64.0% |
| 매칭 후 \|SMD\| | 최대 0.015 (기준 0.1 통과) |
| 단순 격차 | +1.85%p |
| 매칭 후 ATT | −6.53%p |
| 95% CI | −7.83%p ~ −5.11%p |
| 민감도 6종 | −6.47%p ~ −7.65%p · 부호 일치 |

*표 21*


![그림 7](img/fig_09_07.png)


*그림 7*

읽는 법 — 붉은 점이 매칭 전, 파란 점이 매칭 후 |SMD| 임. 점선 0.1 왼쪽으로 모두 들어와야 비교가 성립함.


|  | 쌍수 | 처리군_순감소율 | 대조군_순감소율 | ATT(%p) |
|---|---|---|---|---|
| 골목상권 | 6848 | 0.228 | 0.282 | **-5.39** |
| 관광특구 | 50 | 0.34 | 0.46 | **-12** |
| 발달상권 | 1710 | 0.347 | 0.44 | **-9.3** |
| 전통시장 | 1422 | 0.211 | 0.296 | **-8.51** |

*표 22*

읽는 법 — 유형별로 따로 낸 ATT 임. 쌍이 50개뿐인 관광특구는 참고용이고, 나머지 세 유형에서 부호가 같은지가 판정 근거임.

기준 — 쌍이 100개 이상인 유형만 판정 대상. **네 유형에서 부호가 같은지**가 「한 유형의 산물이 아닌가」의 기준임.


**업종 10종을 각각 처리군으로 두고 같은 설계로 매칭한 결과**

| 업종 | 처리군 칸수 | 매칭 쌍 | 매칭 후 최대 |SMD| | 단순 격차(%p) | ATT(%p) | McNemar p |
|---|---|---|---|---|---|---|
| 한식음식점 | 15,677 | 10,030 | 0.015 | +1.85 | −6.52 | 1.6e-25 |
| 중식음식점 | 3,646 | 3,646 | 0.048 | −3.88 | −0.77 | 0.439 |
| 커피-음료 | 12,046 | 12,039 | 0.020 | +1.00 | −0.22 | 0.702 |
| 제과점 | 4,133 | 4,132 | 0.039 | −3.14 | +0.85 | 0.363 |
| 분식전문점 | 7,570 | 7,566 | 0.056 | −2.06 | +0.90 | 0.189 |
| 양식음식점 | 4,172 | 4,169 | 0.050 | +1.20 | +1.30 | 0.181 |
| 일식음식점 | 3,696 | 3,696 | 0.042 | +0.18 | +1.92 | 0.0559 |
| 패스트푸드점 | 3,901 | 3,901 | 0.061 | −1.63 | +1.97 | 0.0376 |
| 호프-간이주점 | 8,448 | 8,448 | 0.031 | +3.88 | +2.34 | 5.3e-04 |
| 치킨전문점 | 4,186 | 4,145 | 0.101 | −5.31 | +3.35 | 9.9e-05 |

*표 23*

읽는 법 — **색이 칠해진 네 업종은 「규모·유동인구·면적·집객시설이 비슷한 곳끼리 비교해도 위험 차이가 남는 업종」임.** 붉은 셋(치킨전문점 +3.35 · 호프-간이주점 +2.34 · 패스트푸드점 +1.97%p)은 조건을 맞춰도 더 위험한 쪽이고, 초록 하나(한식음식점 −6.52%p)는 오히려 더 안전한 쪽임. **색이 없는 여섯 업종은 조건을 맞추면 차이가 사라지는 업종** — 위험이 업종에서 온 것이 아니라 입지·규모에서 왔다는 뜻임. 색 기준은 McNemar p < 0.05 이고 그 안에서 ATT 부호로 초록·붉은색을 나눔(일식음식점 p=0.0559 는 경계 위라 색이 없으나 크기는 패스트푸드점과 거의 같음). 한식음식점만 결과를 보기 전에 선험적으로 고정한 처리군이고 나머지 아홉은 같은 설계를 반복한 탐색적 결과임(다중비교 보정 없음). 치킨전문점은 매칭 후 최대 |SMD| 0.101 로 균형 기준을 살짝 넘겨 해석에 주의가 필요함.

기준 — 색은 **McNemar p < 0.05** 인 업종에만 칠하고 그 안에서 **ATT 부호**로 초록(음수)·붉은색(양수)을 나눔. 매칭 후 최대 |SMD| 는 0.1 미만이어야 비교가 성립함.


### 해석

따라서 “한식음식점 자체가 위험하다”는 단순 해석은 채택되지 않았음. **단순 비교의 +1.85%p는 「한식이라서」가 아니라 「한식이 큰 상권에 몰려 있어서」 생긴 값이었음.**

다만 ATT는 **짝을 찾을 수 있었던 한식음식점 집단(64%)에서의 평균 차이**이므로 모든 한식음식점에 그대로 일반화할 수 없고, 매칭은 관측된 공변량만 맞추므로 임대료처럼 데이터에 없는 변수의 영향은 남아 있음.

**부호 반전이 한식에서만 나오는 것이 아님.** 치킨전문점은 단순 비교에서 −5.31%p(안전해 보임)였으나 규모를 맞추면 +3.35%p 로 뒤집히고, 패스트푸드점도 −1.63 → +1.97%p 로 바뀜. 즉 **단순 집계로 만든 업종 위험 순위는 방향까지 틀릴 수 있다** — 작은 상권에 몰려 있는 업종은 사건이 드물어 안전해 보이고, 큰 상권에 몰려 있는 업종은 그 반대다.



## 10  사후검정 — 어느 상권유형이 실제로 다른가

**핵심 한 줄 — 전체적으로 차이가 있다는 것만으로는 부족해, Dunn 사후검정으로 상권유형을 두 개씩 비교했음. 전통시장과 골목상권만 뚜렷하게 구분되지 않았음.**

### 왜 이 분석을 했나

Kruskal-Wallis는 “어딘가 다름”까지만 알려줌. 정책적으로는 정확히 어느 유형과 어느 유형 사이가 다른지 알아야 함.

### 핵심 결과

- 상권유형 4개에서 가능한 6쌍 중 5쌍이 Bonferroni 보정 후에도 유의했음.
- 전통시장과 골목상권은 p=0.321로 유의한 차이가 없었음.
- 발달상권은 전통시장·골목상권보다 높았고, 관광특구는 세 유형 모두보다 높았음.
- **p 는 6쌍 중 5쌍을 유의로 판정하지만, 효과크기로 보면 그림이 달라진다** — Cliff δ 가 중간(0.33) 이상인 세 쌍은 모두 관광특구가 걸려 있고 그 유형은 58칸뿐임. 쓸 수 있는 비교는 발달상권↔전통시장 0.252 · 골목상권↔발달상권 0.209 로 「작음~중간」임.



| 항목 | 핵심 수치 / 의미 |
|---|---|
| 순위 | {전통시장 ≈ 골목상권} < 발달상권 < 관광특구 |
| p 기준 판정 | 6쌍 중 5쌍 유의 (전통시장↔골목상권만 p=0.321 비유의) |
| 효과크기 기준 | 중간 이상은 관광특구뿐이고 그 유형은 58칸 · 나머지는 δ 0.13~0.21 「작음」 |
| 실무 단위 | 유형 사이 차이보다 같은 유형 안의 폭이 크므로 개별 상권 |

*표 24*


|  | median | mean | size |
|---|---|---|---|
| 전통시장 | 0.1667 | 0.2176 | 775 |
| 골목상권 | 0.25 | 0.2282 | 2922 |
| 발달상권 | 0.25 | 0.2894 | 1932 |
| 관광특구 | 0.3333 | 0.3779 | 58 |

*표 25*

읽는 법 — 유형별 중앙값·평균·칸수임. 정렬은 중앙값이 낮은 쪽부터임.

기준 — 중앙값·평균·칸수 서술. 중앙값이 같아도 평균 순위가 다를 수 있어 순위 기반 검정이 필요함.


![그림 8](img/fig_10_08.png)


*그림 8*

읽는 법 — 점이 유형별 평균 순감소 비율, 가로선이 95% 신뢰구간임. **신뢰구간이 겹치지 않으면 두 유형이 다르다고 볼 수 있음** — 전통시장과 골목상권은 겹치고, 발달상권은 떨어져 있으며, 관광특구는 구간 자체가 넓음. 오른쪽 라벨이 나머지 세 유형과 비교한 효과크기임.


**네 상권유형을 한 번에 — 각 유형 대 나머지**

| 상권유형 | 칸수 | 평균 순감소 비율(%) | 95% CI | 나머지와 평균차(%p) | Cliff δ | |δ| 판정 |
|---|---|---|---|---|---|---|
| 전통시장 | 775 | 21.8 | 20.8 ~ 22.7 | −3.64 | −0.133 | 무시할 수준 |
| 골목상권 | 2,922 | 22.8 | 22.3 ~ 23.3 | −4.29 | −0.144 | 무시할 수준 |
| 발달상권 | 1,932 | 28.9 | 28.2 ~ 29.7 | +6.11 | +0.210 | 작음 |
| 관광특구 | 58 | 37.8 | 33.9 ~ 41.8 | +13.01 | +0.453 | 중간 |

*표 26*

읽는 법 — 사후검정은 방법론상 쌍별이지만, 읽기 쉽게 **각 유형을 「나머지 세 유형」과 비교해 네 줄로 정리**했음. Cliff δ 는 「그 유형의 칸이 나머지보다 높을 확률 − 그 반대 확률」이고 0.147 작음 / 0.33 중간 / 0.474 큼으로 읽음. **색 기준은 「칸수 100 이상이면서 |δ| 0.147 이상」임** — 관광특구는 δ 0.453 으로 가장 크지만 58칸(독립 상권 6곳)이고 신뢰구간이 33.9~41.8% 로 넓어 제외했고, 그래서 **실제로 쓸 수 있는 유형 중에서는 발달상권(δ 0.210)만 색이 칠해짐**. 나머지 세 유형은 δ 0.13~0.21 로 「작음」 구간이며, **전통시장과 골목상권은 δ 가 −0.133 / −0.144 로 사실상 같은 자리**임.

기준 — **Cliff δ 는 0.147 작음 / 0.33 중간 / 0.474 큼**(Romano 등의 관행값). 확률로 읽으면 δ 0.147 ≈ 57%, 0.210 ≈ 60.5%, 0.453 ≈ 73% 로 한쪽이 더 높을 확률임. 색 기준은 「칸수 100 이상 + |δ| 0.147 이상」.


### 해석

**따라서 「네 유형이 서로 다름」는 진술은 p 가 만든 것이고, 실질적 크기는 작음.** 유형 사이의 차이보다 같은 유형 안의 폭이 크므로, 대응 단위를 유형이 아니라 상권으로 잡아야 한다는 근거가 여기서도 나옴.

